# IICS Folder-to-Folder Mapping & Transformation Comparator

This notebook compares two IICS/IDMC projects using REST APIs.

It will:

- Login to IICS.
- Retrieve assets from Project A and Project B.
- Identify common assets, assets only in A, and assets only in B.
- Match common mapping-oriented assets by relative path and type.
- Export matching assets from both projects using Platform REST API v3.
- Download and unzip the export packages.
- Locate exported JSON definitions.
- Normalize volatile IDs, timestamps, audit metadata, and environment-specific values.
- Compare transformations, fields/ports, expressions, conditions, parameters, and connections.
- Produce a full recursive JSON structural diff as a fallback.
- Export CSV and Excel comparison reports.

The parser is deliberately tolerant of IICS export JSON differences between releases.


## Folder-scoped comparison

The comparison is scoped to:

```text
PROJECT_A / FOLDER_A
vs
PROJECT_B / FOLDER_B
```

All nested subfolders are included. Assets are matched by relative path below the selected folder root plus asset type.


## 1. Install dependencies

In [ ]:
# Uncomment if needed
# %pip install requests pandas openpyxl


## 2. Configuration

In [ ]:
import os

LOGIN_URL = os.getenv(
    "IICS_LOGIN_URL",
    "https://dm-us.informaticacloud.com/ma/api/v2/user/login"
)

USERNAME = os.getenv("IICS_USERNAME", "")
PASSWORD = os.getenv("IICS_PASSWORD", "")

# -------------------------------------------------------------------
# Select project + folder on each side.
# Use "" for FOLDER_A / FOLDER_B to compare the full project.
# -------------------------------------------------------------------

PROJECT_A = "DEV"
FOLDER_A = "Claims/Inbound"

PROJECT_B = "PROD"
FOLDER_B = "Claims/Inbound"

MATCH_BY_RELATIVE_PATH = True

COMPARE_ASSET_TYPES = {
    "MAPPING",
    "DTEMPLATE",
    "MTT",
    "MAPPLET",
    "TASKFLOW",
    "WORKFLOW"
}

INCLUDE_DEPENDENCIES = False
EXPORT_BATCH_SIZE = 20
EXPORT_POLL_SECONDS = 2
EXPORT_TIMEOUT_SECONDS = 1800

OUTPUT_DIR = "iics_folder_comparison"


def join_iics_path(*parts):
    cleaned = []
    for part in parts:
        value = str(part or "").strip().strip("/")
        if value:
            cleaned.append(value)
    return "/".join(cleaned)


ROOT_A = join_iics_path(PROJECT_A, FOLDER_A)
ROOT_B = join_iics_path(PROJECT_B, FOLDER_B)

print("Project A :", PROJECT_A)
print("Folder A  :", FOLDER_A or "<project root>")
print("Root A    :", ROOT_A)
print()
print("Project B :", PROJECT_B)
print("Folder B  :", FOLDER_B or "<project root>")
print("Root B    :", ROOT_B)
print()
print("Output    :", OUTPUT_DIR)


## 3. Imports and helper functions

In [ ]:
from __future__ import annotations

import hashlib
import json
import re
import time
import zipfile
from datetime import datetime
from pathlib import Path, PurePosixPath

import pandas as pd
import requests
from IPython.display import display

REQUEST_TIMEOUT = 90


def safe_json(response):
    try:
        return response.json()
    except Exception as exc:
        raise RuntimeError(
            f"Expected JSON from {response.url}; "
            f"HTTP {response.status_code}: {response.text[:1500]}"
        ) from exc


def check_response(response):
    if not response.ok:
        raise RuntimeError(
            f"HTTP {response.status_code} calling "
            f"{response.request.method} {response.url}\n"
            f"{response.text[:3000]}"
        )
    return response


def normalize_project_path(path):
    path = str(path or "").strip().replace("\\", "/")
    path = re.sub(r"/+", "/", path)
    return path.strip("/")


def normalize_asset_type(value):
    return str(value or "").strip().upper()


def first_nonempty(obj, keys, default=None):
    if not isinstance(obj, dict):
        return default
    for key in keys:
        value = obj.get(key)
        if value not in (None, "", [], {}):
            return value
    return default


def compact_json(value):
    return json.dumps(
        value,
        sort_keys=True,
        ensure_ascii=False,
        separators=(",", ":"),
        default=str
    )


## 4. IICS REST client

In [ ]:
class IICSClient:
    def __init__(self, login_url, username, password):
        self.login_url = login_url.rstrip("/")
        self.username = username
        self.password = password
        self.session_id = None
        self.server_url = None
        self.base_api_url = None
        self.org_id = None

    def login(self):
        payload = {
            "@type": "login",
            "username": self.username,
            "password": self.password
        }

        r = requests.post(
            self.login_url,
            headers={
                "Accept": "application/json",
                "Content-Type": "application/json"
            },
            json=payload,
            timeout=REQUEST_TIMEOUT
        )
        check_response(r)
        data = safe_json(r)

        self.session_id = data.get("icSessionId")
        self.server_url = (data.get("serverUrl") or "").rstrip("/")
        self.base_api_url = (
            data.get("baseApiUrl")
            or data.get("baseAPIUrl")
            or self.server_url
        ).rstrip("/")
        self.org_id = data.get("orgId") or data.get("organizationId")

        if not self.session_id:
            raise RuntimeError("Login response did not return icSessionId.")
        if not self.base_api_url:
            raise RuntimeError(
                "Login response did not return baseApiUrl/serverUrl."
            )

        return data

    @property
    def headers(self):
        if not self.session_id:
            raise RuntimeError("Call login() first.")

        # Different IICS resources/releases use one of these headers.
        return {
            "Accept": "application/json",
            "Content-Type": "application/json",
            "INFA-SESSION-ID": self.session_id,
            "icSessionId": self.session_id
        }

    def get(self, path, params=None, stream=False):
        url = (
            path if str(path).startswith("http")
            else f"{self.base_api_url}/{str(path).lstrip('/')}"
        )
        r = requests.get(
            url,
            headers=self.headers,
            params=params,
            timeout=REQUEST_TIMEOUT,
            stream=stream
        )
        return check_response(r)

    def post(self, path, payload):
        url = (
            path if str(path).startswith("http")
            else f"{self.base_api_url}/{str(path).lstrip('/')}"
        )
        r = requests.post(
            url,
            headers=self.headers,
            json=payload,
            timeout=REQUEST_TIMEOUT
        )
        return check_response(r)

    @staticmethod
    def _extract_objects(payload):
        if isinstance(payload, list):
            return payload
        if not isinstance(payload, dict):
            return []

        for key in ("objects", "entries", "items", "results"):
            value = payload.get(key)
            if isinstance(value, list):
                return value

        if "id" in payload and (
            "name" in payload or
            "path" in payload or
            "location" in payload
        ):
            return [payload]

        return []

    @staticmethod
    def object_path(obj):
        path = first_nonempty(
            obj,
            ["path", "fullPath", "objectPath", "location", "sourcePath"],
            default=""
        )
        name = first_nonempty(
            obj,
            ["name", "objectName", "assetName"],
            default=""
        )

        path = str(path or "").replace("\\", "/").strip()

        if path and name and obj.get("location") == path:
            clean = path.rstrip("/")
            if not clean.endswith("/" + str(name)):
                path = clean + "/" + str(name)

        if not path:
            path = str(name or "")

        return "/" + normalize_project_path(path)

    @staticmethod
    def path_is_under_project(full_path, project_path):
        fp = normalize_project_path(full_path).casefold()
        pp = normalize_project_path(project_path).casefold()
        return fp == pp or fp.startswith(pp + "/")

    def list_objects(self, project_path):
        project_path = normalize_project_path(project_path)
        endpoint = "/public/core/v3/objects"

        # POD/release query syntax can differ, so try common forms and
        # always enforce project scoping locally.
        attempts = [
            {"q": f"location=='{project_path}'"},
            {"q": f"location=='/{project_path}'"},
            {}
        ]

        last_error = None

        for params in attempts:
            try:
                payload = safe_json(self.get(endpoint, params=params))
                rows = self._extract_objects(payload)

                scoped = [
                    obj for obj in rows
                    if self.path_is_under_project(
                        self.object_path(obj),
                        project_path
                    )
                ]

                if scoped:
                    return scoped

                if not rows:
                    return []

            except Exception as exc:
                last_error = exc

        if last_error:
            raise last_error

        return []

    def start_export(self, objects, name, include_dependencies=False):
        export_objects = []

        for obj in objects:
            object_id = first_nonempty(
                obj,
                ["id", "objectId", "assetId"]
            )
            if object_id:
                item = {"id": object_id}
                if include_dependencies:
                    item["includeDependencies"] = True
                export_objects.append(item)

        if not export_objects:
            raise ValueError("No object IDs supplied for export.")

        payload = {
            "name": name,
            "objects": export_objects
        }

        try:
            data = safe_json(
                self.post("/public/core/v3/export", payload)
            )
        except RuntimeError:
            if not include_dependencies:
                raise

            payload["objects"] = [
                {"id": x["id"]}
                for x in export_objects
            ]
            data = safe_json(
                self.post("/public/core/v3/export", payload)
            )

        export_id = first_nonempty(
            data,
            ["id", "jobId", "exportId"]
        )

        if not export_id:
            raise RuntimeError(
                f"Export started but no export job ID returned: {data}"
            )

        return export_id

    def export_status(self, export_id):
        return safe_json(
            self.get(
                f"/public/core/v3/export/{export_id}",
                params={"expand": "objects"}
            )
        )

    @staticmethod
    def _export_state(data):
        if not isinstance(data, dict):
            return "", ""

        status_obj = data.get("status")

        if isinstance(status_obj, dict):
            state = status_obj.get("state") or status_obj.get("status") or ""
            message = status_obj.get("message") or ""
            return str(state).strip().upper(), str(message)

        state = data.get("state") or data.get("jobStatus") or status_obj or ""
        message = data.get("message") or ""
        return str(state).strip().upper(), str(message)

    def wait_for_export(self, export_id):
        deadline = time.time() + EXPORT_TIMEOUT_SECONDS
        previous_display = None
        last_data = None

        success_states = {
            "SUCCESSFUL",
            "SUCCESS",
            "COMPLETED",
            "COMPLETED_SUCCESSFULLY"
        }

        failed_states = {
            "FAILED",
            "FAILURE",
            "ERROR",
            "CANCELLED",
            "CANCELED",
            "ABORTED"
        }

        while time.time() < deadline:
            data = self.export_status(export_id)
            last_data = data

            state, message = self._export_state(data)

            display_state = (state, message)
            if display_state != previous_display:
                print(
                    f"  Export {export_id}: state={state or '<blank>'}"
                    + (f" | {message}" if message else "")
                )
                previous_display = display_state

            if state in success_states:
                return data

            if state in failed_states:
                objects = data.get("objects", []) if isinstance(data, dict) else []
                failed_objects = []

                if isinstance(objects, list):
                    for obj in objects:
                        obj_state, obj_message = self._export_state(obj)
                        if obj_state in failed_states:
                            failed_objects.append({
                                "name": obj.get("name"),
                                "path": obj.get("path"),
                                "state": obj_state,
                                "message": obj_message
                            })

                detail = (
                    f"\nFailed objects: {failed_objects[:20]}"
                    if failed_objects else ""
                )

                raise RuntimeError(
                    f"IICS export {export_id} failed. "
                    f"state={state}, message={message}"
                    f"{detail}"
                )

            time.sleep(EXPORT_POLL_SECONDS)

        state, message = self._export_state(last_data or {})

        raise TimeoutError(
            f"Timed out waiting for export {export_id} after "
            f"{EXPORT_TIMEOUT_SECONDS} seconds. "
            f"Last state={state or '<blank>'}; "
            f"message={message or '<none>'}. "
            "The export job has NOT been cancelled; query the same export ID "
            "again with client.export_status(export_id)."
        )

    def download_export(self, export_id, destination):
        destination = Path(destination)
        destination.parent.mkdir(parents=True, exist_ok=True)

        r = self.get(
            f"/public/core/v3/export/{export_id}/package",
            stream=True
        )

        with destination.open("wb") as fh:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    fh.write(chunk)

        return destination


## Validate comparison roots

In [ ]:
assert ROOT_A, "ROOT_A cannot be empty."
assert ROOT_B, "ROOT_B cannot be empty."

print("Comparing folder trees:")
print(" A:", ROOT_A)
print(" B:", ROOT_B)


## 5. Login and retrieve both project inventories

In [ ]:
if not USERNAME or not PASSWORD:
    raise ValueError(
        "Set IICS_USERNAME and IICS_PASSWORD environment variables "
        "or populate USERNAME/PASSWORD in the configuration cell."
    )

client = IICSClient(
    LOGIN_URL,
    USERNAME,
    PASSWORD
)
client.login()

print("Connected to :", client.base_api_url)
print("Organization :", client.org_id)


def relative_asset_path(full_path, project_path):
    fp = normalize_project_path(full_path)
    pp = normalize_project_path(project_path)

    if fp.casefold() == pp.casefold():
        return ""

    prefix = pp + "/"
    if fp.casefold().startswith(prefix.casefold()):
        return fp[len(prefix):]

    return fp


def normalize_inventory(objects, project_path):
    rows = []

    for obj in objects:
        full_path = client.object_path(obj)
        rel_path = relative_asset_path(
            full_path,
            project_path
        )

        rows.append({
            "id": first_nonempty(
                obj,
                ["id", "objectId", "assetId"],
                default=""
            ),
            "name": first_nonempty(
                obj,
                ["name", "objectName", "assetName"],
                default=PurePosixPath(rel_path).name
            ),
            "type": normalize_asset_type(
                first_nonempty(
                    obj,
                    ["type", "objectType", "assetType"],
                    default=""
                )
            ),
            "full_path": full_path,
            "relative_path": rel_path,
            "updated_at": first_nonempty(
                obj,
                [
                    "updateTime",
                    "updatedAt",
                    "lastUpdated",
                    "modifiedTime"
                ],
                default=""
            )
        })

    return pd.DataFrame(rows)


inventory_a = normalize_inventory(
    client.list_objects(ROOT_A),
    ROOT_A
)
inventory_b = normalize_inventory(
    client.list_objects(ROOT_B),
    ROOT_B
)

print("Folder A assets:", len(inventory_a))
print("Folder B assets:", len(inventory_b))

display(inventory_a.head(20))
display(inventory_b.head(20))


## 6. Match common assets

In [ ]:
def make_match_key(row):
    asset_type = normalize_asset_type(row["type"])

    if MATCH_BY_RELATIVE_PATH:
        identity = str(
            row["relative_path"]
        ).strip("/").casefold()
    else:
        identity = str(row["name"]).casefold()

    return f"{asset_type}|{identity}"


inventory_a["match_key"] = (
    inventory_a.apply(make_match_key, axis=1)
    if not inventory_a.empty
    else pd.Series(dtype="object")
)
inventory_b["match_key"] = (
    inventory_b.apply(make_match_key, axis=1)
    if not inventory_b.empty
    else pd.Series(dtype="object")
)

a_keys = set(inventory_a["match_key"])
b_keys = set(inventory_b["match_key"])

common_keys = a_keys & b_keys

common_assets = (
    inventory_a[
        inventory_a["match_key"].isin(common_keys)
    ]
    .merge(
        inventory_b[
            inventory_b["match_key"].isin(common_keys)
        ],
        on="match_key",
        suffixes=("_a", "_b")
    )
)

only_a = inventory_a[
    ~inventory_a["match_key"].isin(common_keys)
].copy()

only_b = inventory_b[
    ~inventory_b["match_key"].isin(common_keys)
].copy()

if COMPARE_ASSET_TYPES is None:
    common_to_compare = common_assets.copy()
else:
    allowed = {
        normalize_asset_type(x)
        for x in COMPARE_ASSET_TYPES
    }
    common_to_compare = common_assets[
        common_assets["type_a"].isin(allowed)
    ].copy()

print("Common assets       :", len(common_assets))
print("Only in Project A   :", len(only_a))
print("Only in Project B   :", len(only_b))
print("Selected deep compare:", len(common_to_compare))

display(
    common_to_compare[
        [
            "relative_path_a",
            "type_a",
            "id_a",
            "id_b"
        ]
    ].head(100)
)


## 7. Export matched assets from both projects

In [ ]:
output_root = Path(OUTPUT_DIR)
raw_dir = output_root / "raw_exports"
extract_dir = output_root / "extracted"

raw_dir.mkdir(parents=True, exist_ok=True)
extract_dir.mkdir(parents=True, exist_ok=True)


def batch_dataframe(df, size):
    for start in range(0, len(df), size):
        yield df.iloc[start:start + size]


def recursively_extract_archives(root, max_passes=6):
    """
    IICS export ZIPs can contain asset packages such as:
        mapping_name.DTEMPLATE
        task_name.MTT

    These files are often ZIP archives even though the extension is
    not '.zip'. Recursively unpack every ZIP-format file so that
    mapping payload files such as bin/@3.bin or bin/@2.bin become
    visible to the comparator.
    """
    root = Path(root)
    extracted = set()

    for _ in range(max_passes):
        found_new = False

        for path in list(root.rglob("*")):
            if not path.is_file():
                continue

            key = str(path.resolve())
            if key in extracted:
                continue

            try:
                is_archive = zipfile.is_zipfile(path)
            except Exception:
                is_archive = False

            if not is_archive:
                continue

            extracted.add(key)

            # Outer export ZIP is already expanded by export_side.
            # Nested archive extraction directory keeps the original
            # asset filename so mapping association is deterministic.
            target = path.parent / f"{path.name}__unzipped"

            if target.exists():
                continue

            target.mkdir(parents=True, exist_ok=True)

            try:
                with zipfile.ZipFile(path, "r") as zf:
                    zf.extractall(target)

                print(
                    "    Nested IICS package:",
                    path.name,
                    "->",
                    target.name
                )
                found_new = True

            except Exception as exc:
                print(
                    "    WARNING: could not unpack",
                    path,
                    ":",
                    exc
                )

        if not found_new:
            break


def export_side(df, side):
    result_dirs = []

    id_col = f"id_{side.lower()}"

    for batch_no, batch in enumerate(
        batch_dataframe(df, EXPORT_BATCH_SIZE),
        start=1
    ):
        objects = [
            {"id": row[id_col]}
            for _, row in batch.iterrows()
            if row[id_col]
        ]

        if not objects:
            continue

        stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        job_name = f"compare_{side}_{batch_no}_{stamp}"

        print(
            f"Exporting {side} batch {batch_no}: "
            f"{len(objects)} assets"
        )

        export_id = client.start_export(
            objects,
            name=job_name,
            include_dependencies=INCLUDE_DEPENDENCIES
        )

        client.wait_for_export(export_id)

        zip_path = (
            raw_dir /
            f"{job_name}_{export_id}.zip"
        )
        client.download_export(
            export_id,
            zip_path
        )

        target = (
            extract_dir /
            f"{side}_{batch_no}_{export_id}"
        )
        target.mkdir(
            parents=True,
            exist_ok=True
        )

        with zipfile.ZipFile(
            zip_path,
            "r"
        ) as zf:
            zf.extractall(target)

        # Important: DTEMPLATE / MTT etc. may themselves be ZIP files.
        recursively_extract_archives(target)

        result_dirs.append(target)

        print("  ZIP      :", zip_path)
        print("  Extracted:", target)

    return result_dirs


dirs_a = export_side(
    common_to_compare,
    "A"
)
dirs_b = export_side(
    common_to_compare,
    "B"
)


### Export timeout troubleshooting

The revised polling logic reads the actual IICS export state from `status.state`.

Defaults are now:
- **20 assets per export batch**
- **30-minute timeout per batch**

If an export is still running when the timeout is reached, copy its export ID and inspect that existing job rather than starting a duplicate export.


In [ ]:
# Optional: inspect an existing export job
# EXPORT_ID_TO_CHECK = "paste-export-id-here"
# status_payload = client.export_status(EXPORT_ID_TO_CHECK)
# display(status_payload)
# print("Resolved state:", client._export_state(status_payload))


## 8. Index exported JSON files and associate them with assets

In [ ]:
def load_json_like(path):
    """
    Load JSON from ordinary .json files OR IICS @*.bin mapping payloads.

    Some DTEMPLATE exports store the mapping definition in files such
    as bin/@3.bin or bin/@2.bin. These can contain JSON despite the
    .bin extension.
    """
    path = Path(path)

    for encoding in ("utf-8", "utf-8-sig"):
        try:
            text = path.read_text(
                encoding=encoding,
                errors="strict"
            )
        except Exception:
            continue

        stripped = text.lstrip("\ufeff\r\n\t ")

        if not stripped.startswith(("{", "[")):
            continue

        try:
            return json.loads(stripped)
        except Exception:
            continue

    return None


def index_mapping_payload_files(root_dirs):
    rows = []

    for root in root_dirs:
        root = Path(root)

        for path in root.rglob("*"):
            if not path.is_file():
                continue

            suffix = path.suffix.casefold()
            name_cf = path.name.casefold()

            candidate = (
                suffix == ".json"
                or suffix == ".bin"
                or name_cf.startswith("@")
            )

            if not candidate:
                continue

            obj = load_json_like(path)

            if obj is None:
                continue

            rel = str(
                path.relative_to(root)
            ).replace("\\", "/")

            # Content classification
            is_bin = (
                suffix == ".bin"
                or path.name.startswith("@")
            )

            is_mapping_bin = (
                is_bin
                and "/bin/" in f"/{rel.casefold()}/"
            )

            bin_rank = 0
            if name_cf == "@3.bin":
                bin_rank = 3
            elif name_cf == "@2.bin":
                bin_rank = 2
            elif name_cf == "@1.bin":
                bin_rank = 1
            elif is_bin:
                bin_rank = 0.5

            rows.append({
                "file": str(path),
                "file_name": path.name,
                "relative_file": rel,
                "size": path.stat().st_size,
                "json": obj,
                "text": compact_json(obj),
                "is_bin_payload": is_bin,
                "is_mapping_bin": is_mapping_bin,
                "bin_rank": bin_rank
            })

    if not rows:
        return pd.DataFrame(
            columns=[
                "file", "file_name", "relative_file", "size",
                "json", "text", "is_bin_payload",
                "is_mapping_bin", "bin_rank"
            ]
        )

    return pd.DataFrame(rows)


payload_files_a = index_mapping_payload_files(dirs_a)
payload_files_b = index_mapping_payload_files(dirs_b)

print("JSON/JSON-like payload files A:", len(payload_files_a))
print("JSON/JSON-like payload files B:", len(payload_files_b))

print(
    "Mapping bin payloads A:",
    int(payload_files_a["is_mapping_bin"].sum())
    if not payload_files_a.empty else 0
)
print(
    "Mapping bin payloads B:",
    int(payload_files_b["is_mapping_bin"].sum())
    if not payload_files_b.empty else 0
)


def score_candidate(
    file_row,
    asset_id,
    asset_name,
    relative_path,
    asset_type
):
    text = file_row["text"].casefold()
    filename = file_row["file_name"].casefold()
    relfile = file_row["relative_file"].casefold()

    aid = str(asset_id or "").casefold()
    name = str(asset_name or "").casefold()
    rel = str(relative_path or "").strip("/").casefold()
    atype = str(asset_type or "").upper()

    score = 0.0

    # Strongly prefer DTEMPLATE @*.bin mapping definitions.
    if atype in {"DTEMPLATE", "MAPPING"}:
        if bool(file_row.get("is_mapping_bin", False)):
            score += 500

        score += float(
            file_row.get("bin_rank", 0) or 0
        ) * 25

        if ".dtemplate__unzipped" in relfile:
            score += 250

    # For mapping tasks, task JSON should win over unrelated mapping bins.
    if atype == "MTT":
        if "mtt__unzipped" in relfile:
            score += 250
        if filename.endswith(".json"):
            score += 30

    if aid:
        if aid in text:
            score += 100
        if aid in filename or aid in relfile:
            score += 150

    if name:
        # Nested extraction directory includes e.g.
        # m_claim.DTEMPLATE__unzipped.
        if name in relfile:
            score += 200
        if name in filename:
            score += 80
        if f'"name":"{name}"' in text:
            score += 60
        elif name in text:
            score += 20

    if rel and rel in text:
        score += 40

    # Prefer substantive payloads over tiny descriptor JSON.
    score += min(
        file_row["size"] / 50000.0,
        30
    )

    return score


def choose_payload(
    files_df,
    asset_id,
    asset_name,
    relative_path,
    asset_type
):
    if files_df.empty:
        return None, 0.0

    scored = []

    for idx, row in files_df.iterrows():
        score = score_candidate(
            row,
            asset_id,
            asset_name,
            relative_path,
            asset_type
        )

        if score > 0:
            scored.append(
                (score, row["size"], idx)
            )

    if not scored:
        return None, 0.0

    scored.sort(
        reverse=True
    )

    best_score, _, best_idx = scored[0]

    return (
        files_df.loc[best_idx],
        best_score
    )


resolved = []

for _, row in common_to_compare.iterrows():
    a_file, a_score = choose_payload(
        payload_files_a,
        row["id_a"],
        row["name_a"],
        row["relative_path_a"],
        row["type_a"]
    )

    b_file, b_score = choose_payload(
        payload_files_b,
        row["id_b"],
        row["name_b"],
        row["relative_path_b"],
        row["type_b"]
    )

    resolved.append({
        "match_key": row["match_key"],
        "relative_path": row["relative_path_a"],
        "asset_type": row["type_a"],
        "asset_name": row["name_a"],
        "id_a": row["id_a"],
        "id_b": row["id_b"],
        "json_file_a": (
            None
            if a_file is None
            else a_file["file"]
        ),
        "json_file_b": (
            None
            if b_file is None
            else b_file["file"]
        ),
        "payload_kind_a": (
            None
            if a_file is None
            else (
                "MAPPING_BIN"
                if a_file["is_mapping_bin"]
                else "JSON"
            )
        ),
        "payload_kind_b": (
            None
            if b_file is None
            else (
                "MAPPING_BIN"
                if b_file["is_mapping_bin"]
                else "JSON"
            )
        ),
        "score_a": a_score,
        "score_b": b_score,
        "json_a": (
            None
            if a_file is None
            else a_file["json"]
        ),
        "json_b": (
            None
            if b_file is None
            else b_file["json"]
        )
    })

asset_json_map = pd.DataFrame(resolved)

print(
    "\nResolved mapping payloads "
    "(DTEMPLATE mappings should normally show MAPPING_BIN):"
)

display(
    asset_json_map[
        [
            "relative_path",
            "asset_type",
            "payload_kind_a",
            "payload_kind_b",
            "json_file_a",
            "json_file_b",
            "score_a",
            "score_b"
        ]
    ]
)


## 9. Normalize volatile IICS metadata

In [ ]:
IGNORE_KEYS = {
    "id",
    "objectid",
    "assetid",
    "uuid",
    "guid",
    "createdby",
    "createdat",
    "createtime",
    "createdtime",
    "updatedby",
    "updatedat",
    "updatetime",
    "modifiedby",
    "modifiedat",
    "lastmodified",
    "lastupdated",
    "revision",
    "revisionid",
    "versionid",
    "etag",
    "checksum",
    "hash",
    "orgid",
    "organizationid",
    "packageid",
    "exportid"
}


def ignore_key(key):
    norm = (
        str(key)
        .replace("_", "")
        .replace("-", "")
        .casefold()
    )

    return (
        norm in IGNORE_KEYS
        or norm.endswith("timestamp")
        or "lastmodified" in norm
    )


def normalize_json(value):
    if isinstance(value, dict):
        result = {}

        for key, child in value.items():
            if ignore_key(key):
                continue
            result[key] = normalize_json(child)

        return {
            key: result[key]
            for key in sorted(
                result,
                key=lambda x: str(x).casefold()
            )
        }

    if isinstance(value, list):
        items = [
            normalize_json(x)
            for x in value
        ]

        preserve_order = any(
            isinstance(x, dict)
            and any(
                str(k).casefold()
                in {
                    "order",
                    "index",
                    "sequence",
                    "position",
                    "ordinal"
                }
                for k in x
            )
            for x in value
        )

        if preserve_order:
            return items

        try:
            return sorted(
                items,
                key=compact_json
            )
        except Exception:
            return items

    if isinstance(value, str):
        return "\n".join(
            line.rstrip()
            for line in value.replace(
                "\r\n",
                "\n"
            ).split("\n")
        ).strip()

    return value


## 10. Semantic parser for transformations, fields, expressions and conditions

In [ ]:
NAME_KEYS = {
    "name",
    "transformationname",
    "objectname",
    "fieldname",
    "portname",
    "parametername",
    "stepname",
    "label"
}

TRANSFORMATION_TYPE_KEYS = {
    "transformationtype",
    "transformtype",
    "transformation",
    "type"
}

TRANSFORMATION_HINTS = {
    "expression",
    "filter",
    "lookup",
    "joiner",
    "aggregator",
    "router",
    "sorter",
    "rank",
    "sequence",
    "source",
    "target",
    "normalizer",
    "union",
    "java",
    "sql",
    "storedprocedure",
    "webservice",
    "hierarchy",
    "datamasking",
    "data masking"
}

EXPRESSION_KEYS = {
    "expression",
    "expr",
    "formula",
    "sql",
    "sqlquery",
    "query"
}

CONDITION_KEYS = {
    "condition",
    "filtercondition",
    "joincondition",
    "lookupcondition",
    "routercondition",
    "conditionexpression",
    "criteria"
}

FIELD_KEYS = {
    "field",
    "fields",
    "port",
    "ports",
    "inputfield",
    "outputfield",
    "sourcefield",
    "targetfield",
    "column",
    "columns",
    "fieldmapping",
    "fieldmappings"
}

PARAMETER_KEYS = {
    "parameter",
    "parameters",
    "parametername",
    "param"
}

CONNECTION_KEYS = {
    "connection",
    "connectionname",
    "runtimeenvironment",
    "runtimeenvironmentname"
}


def recursive_nodes(value, path="$"):
    yield path, value

    if isinstance(value, dict):
        for key, child in value.items():
            yield from recursive_nodes(
                child,
                f"{path}.{key}"
            )

    elif isinstance(value, list):
        for idx, child in enumerate(value):
            yield from recursive_nodes(
                child,
                f"{path}[{idx}]"
            )


def find_name(obj):
    if not isinstance(obj, dict):
        return ""

    for key, value in obj.items():
        if (
            str(key).casefold() in NAME_KEYS
            and isinstance(
                value,
                (str, int, float)
            )
        ):
            return str(value)

    return ""


def find_type(obj):
    if not isinstance(obj, dict):
        return ""

    candidates = []

    for key, value in obj.items():
        kn = (
            str(key)
            .replace("_", "")
            .casefold()
        )

        if (
            kn in TRANSFORMATION_TYPE_KEYS
            and isinstance(
                value,
                (str, int, float)
            )
        ):
            candidates.append(
                str(value)
            )

    for candidate in candidates:
        text = (
            candidate
            .casefold()
            .replace("_", " ")
        )

        if any(
            hint in text
            for hint in TRANSFORMATION_HINTS
        ):
            return candidate

    return (
        candidates[0]
        if candidates
        else ""
    )


def semantic_records(asset_json):
    columns = [
        "category",
        "name",
        "type",
        "property",
        "value",
        "path"
    ]

    if asset_json is None:
        return pd.DataFrame(
            columns=columns
        )

    source = normalize_json(
        asset_json
    )
    rows = []

    exp_keys = {
        x.replace("_", "")
        for x in EXPRESSION_KEYS
    }
    condition_keys = {
        x.replace("_", "")
        for x in CONDITION_KEYS
    }
    field_keys = {
        x.replace("_", "")
        for x in FIELD_KEYS
    }
    parameter_keys = {
        x.replace("_", "")
        for x in PARAMETER_KEYS
    }
    connection_keys = {
        x.replace("_", "")
        for x in CONNECTION_KEYS
    }

    for path, node in recursive_nodes(source):
        if not isinstance(node, dict):
            continue

        name = find_name(node)
        node_type = find_type(node)

        if name and node_type:
            type_text = (
                node_type
                .casefold()
                .replace("_", " ")
            )

            if any(
                hint in type_text
                for hint in TRANSFORMATION_HINTS
            ):
                rows.append({
                    "category": "TRANSFORMATION",
                    "name": name,
                    "type": node_type,
                    "property": "definition",
                    "value": compact_json(node),
                    "path": path
                })

        for key, value in node.items():
            key_norm = (
                str(key)
                .replace("_", "")
                .casefold()
            )

            category = None

            if key_norm in exp_keys:
                category = "EXPRESSION"
            elif key_norm in condition_keys:
                category = "CONDITION"
            elif key_norm in field_keys:
                category = "FIELD_OR_MAPPING"
            elif key_norm in parameter_keys:
                category = "PARAMETER"
            elif key_norm in connection_keys:
                category = "CONNECTION"

            if category is None:
                continue

            if isinstance(
                value,
                (dict, list)
            ):
                value_repr = compact_json(
                    value
                )
            else:
                value_repr = str(value)

            rows.append({
                "category": category,
                "name": (
                    name
                    or PurePosixPath(
                        path.replace(".", "/")
                    ).name
                ),
                "type": node_type,
                "property": str(key),
                "value": value_repr,
                "path": path
            })

    if not rows:
        return pd.DataFrame(
            columns=columns
        )

    return (
        pd.DataFrame(rows)
        .drop_duplicates(
            subset=[
                "category",
                "name",
                "type",
                "property",
                "value"
            ]
        )
        .reset_index(drop=True)
    )


## 11. Compare semantic components and full JSON structure

In [ ]:
def semantic_diff(df_a, df_b):
    key_cols = [
        "category",
        "name",
        "type",
        "property"
    ]

    if df_a.empty and df_b.empty:
        return pd.DataFrame(
            columns=key_cols + [
                "value_a",
                "value_b",
                "change_type"
            ]
        )

    def collapse(df, value_name):
        if df.empty:
            return pd.DataFrame(
                columns=key_cols + [value_name]
            )

        return (
            df.groupby(
                key_cols,
                dropna=False
            )["value"]
            .apply(
                lambda values:
                " || ".join(
                    sorted(
                        set(
                            map(str, values)
                        )
                    )
                )
            )
            .reset_index(
                name=value_name
            )
        )

    a = collapse(
        df_a,
        "value_a"
    )
    b = collapse(
        df_b,
        "value_b"
    )

    merged = a.merge(
        b,
        on=key_cols,
        how="outer",
        indicator=True
    )

    def classify(row):
        if row["_merge"] == "left_only":
            return "ONLY_IN_A"
        if row["_merge"] == "right_only":
            return "ONLY_IN_B"
        if row["value_a"] != row["value_b"]:
            return "CHANGED"
        return "SAME"

    merged["change_type"] = merged.apply(
        classify,
        axis=1
    )

    return (
        merged[
            merged["change_type"] != "SAME"
        ]
        .drop(columns="_merge")
        .reset_index(drop=True)
    )


def flatten_json(value, path="$", result=None):
    if result is None:
        result = {}

    if isinstance(value, dict):
        if not value:
            result[path] = {}

        for key in sorted(
            value,
            key=lambda x: str(x).casefold()
        ):
            flatten_json(
                value[key],
                f"{path}.{key}",
                result
            )

    elif isinstance(value, list):
        if not value:
            result[path] = []

        for idx, child in enumerate(value):
            flatten_json(
                child,
                f"{path}[{idx}]",
                result
            )

    else:
        result[path] = value

    return result


def structural_diff(json_a, json_b):
    a = flatten_json(
        normalize_json(json_a)
    )
    b = flatten_json(
        normalize_json(json_b)
    )

    rows = []

    for path in sorted(
        set(a) | set(b)
    ):
        if path not in b:
            change = "ONLY_IN_A"
        elif path not in a:
            change = "ONLY_IN_B"
        elif a[path] != b[path]:
            change = "CHANGED"
        else:
            continue

        rows.append({
            "json_path": path,
            "change_type": change,
            "value_a": a.get(path),
            "value_b": b.get(path)
        })

    return pd.DataFrame(rows)


summary_rows = []
semantic_frames = []
structural_frames = []

for _, asset in asset_json_map.iterrows():
    json_a = asset["json_a"]
    json_b = asset["json_b"]

    if json_a is None or json_b is None:
        summary_rows.append({
            "relative_path": asset["relative_path"],
            "asset_type": asset["asset_type"],
            "asset_name": asset["asset_name"],
            "comparison_status": "JSON_NOT_FOUND",
            "semantic_differences": None,
            "structural_differences": None
        })
        continue

    sem_a = semantic_records(
        json_a
    )
    sem_b = semantic_records(
        json_b
    )

    sem = semantic_diff(
        sem_a,
        sem_b
    )

    raw = structural_diff(
        json_a,
        json_b
    )

    if not sem.empty:
        sem.insert(
            0,
            "relative_path",
            asset["relative_path"]
        )
        sem.insert(
            1,
            "asset_type",
            asset["asset_type"]
        )
        semantic_frames.append(
            sem
        )

    if not raw.empty:
        raw.insert(
            0,
            "relative_path",
            asset["relative_path"]
        )
        raw.insert(
            1,
            "asset_type",
            asset["asset_type"]
        )
        structural_frames.append(
            raw
        )

    identical = (
        compact_json(
            normalize_json(json_a)
        )
        ==
        compact_json(
            normalize_json(json_b)
        )
    )

    summary_rows.append({
        "relative_path": asset["relative_path"],
        "asset_type": asset["asset_type"],
        "asset_name": asset["asset_name"],
        "comparison_status": (
            "IDENTICAL"
            if identical
            else "DIFFERENT"
        ),
        "semantic_differences": len(sem),
        "structural_differences": len(raw)
    })


asset_comparison_summary = pd.DataFrame(
    summary_rows
)

semantic_differences = (
    pd.concat(
        semantic_frames,
        ignore_index=True
    )
    if semantic_frames
    else pd.DataFrame()
)

structural_differences = (
    pd.concat(
        structural_frames,
        ignore_index=True
    )
    if structural_frames
    else pd.DataFrame()
)

if semantic_differences.empty:
    transformation_differences = pd.DataFrame()
    field_mapping_differences = pd.DataFrame()
else:
    transformation_differences = (
        semantic_differences[
            semantic_differences["category"]
            == "TRANSFORMATION"
        ].copy()
    )

    field_mapping_differences = (
        semantic_differences[
            semantic_differences["category"].isin(
                [
                    "FIELD_OR_MAPPING",
                    "EXPRESSION",
                    "CONDITION"
                ]
            )
        ].copy()
    )

display(
    asset_comparison_summary
)

print(
    "Transformation differences:",
    len(transformation_differences)
)
print(
    "Field/expression/condition differences:",
    len(field_mapping_differences)
)


## Dedicated Field Mapping & Lineage Comparison

This section reconstructs field-level relationships from exported mapping JSON.

It looks for:

- source and target fields,
- transformation input/output ports,
- explicit field mapping objects,
- connector/link objects,
- expressions attached to output ports,
- datatype, precision, scale and length differences,
- missing mappings in either environment,
- changed source-to-target relationships.

Because IICS export schemas can vary by release, the parser uses multiple key patterns and recursive discovery rather than one hard-coded JSON path.


In [ ]:

# -------------------------------------------------------------------
# Dedicated field mapping / lineage parser
# -------------------------------------------------------------------

FIELD_NAME_KEYS = {
    "fieldname", "name", "portname", "columnname", "targetfield",
    "sourcefield", "inputfield", "outputfield"
}

DATATYPE_KEYS = {
    "datatype", "dataType", "type", "nativeType", "fieldType"
}

PRECISION_KEYS = {
    "precision", "length", "size", "maxLength"
}

SCALE_KEYS = {
    "scale", "decimalScale"
}

EXPR_KEYS = {
    "expression", "expr", "formula", "expressionString"
}

DIRECTION_KEYS = {
    "direction", "portType", "ioType", "fieldDirection"
}

TRANSFORM_NAME_KEYS = {
    "transformationName", "transformName", "instanceName",
    "objectName", "name"
}

LINK_SOURCE_KEYS = {
    "source", "sourceField", "from", "fromField", "input",
    "inputField", "srcField", "sourcePort"
}

LINK_TARGET_KEYS = {
    "target", "targetField", "to", "toField", "output",
    "outputField", "tgtField", "targetPort"
}

LINK_CONTAINER_HINTS = {
    "fieldmapping", "fieldmappings", "mapping", "mappings",
    "connector", "connectors", "link", "links", "connection",
    "connections", "wire", "wires"
}


def _ci_get(obj, candidate_keys):
    if not isinstance(obj, dict):
        return None

    lookup = {str(k).casefold(): k for k in obj.keys()}

    for candidate in candidate_keys:
        key = lookup.get(str(candidate).casefold())
        if key is not None:
            return obj.get(key)

    return None


def _scalar_text(value):
    if value is None:
        return ""

    if isinstance(value, (str, int, float, bool)):
        return str(value)

    if isinstance(value, dict):
        for key in (
            "name", "fieldName", "portName", "columnName",
            "path", "qualifiedName", "objectName"
        ):
            v = _ci_get(value, {key})
            if isinstance(v, (str, int, float)):
                return str(v)

    return compact_json(value)


def _qualified_field(value):
    """
    Convert a field/link endpoint object into a stable textual identity.

    Examples:
      SRC_CLAIM.CLAIM_ID
      EXP_CLAIM.CLAIM_ID
      TGT_CLAIM.CLAIM_ID
    """
    if value is None:
        return ""

    if isinstance(value, str):
        return value.strip()

    if not isinstance(value, dict):
        return str(value)

    transform = _ci_get(
        value,
        {
            "transformationName", "transformName", "instanceName",
            "objectName", "parentName", "groupName"
        }
    )

    field = _ci_get(
        value,
        {
            "fieldName", "portName", "columnName", "name",
            "sourceField", "targetField"
        }
    )

    path = _ci_get(
        value,
        {"path", "qualifiedName", "fieldPath"}
    )

    if path:
        return str(path)

    if transform and field:
        return f"{transform}.{field}"

    if field:
        return str(field)

    return compact_json(value)


def _extract_type_attrs(obj):
    datatype = _ci_get(
        obj,
        {"dataType", "datatype", "nativeType", "fieldType"}
    )
    precision = _ci_get(
        obj,
        {"precision", "length", "size", "maxLength"}
    )
    scale = _ci_get(
        obj,
        {"scale", "decimalScale"}
    )

    return (
        "" if datatype is None else str(datatype),
        "" if precision is None else str(precision),
        "" if scale is None else str(scale),
    )


def extract_fields(asset_json):
    """
    Collect field / port definitions with contextual transformation names.
    """
    rows = []

    if asset_json is None:
        return pd.DataFrame()

    normalized = normalize_json(asset_json)

    for path, node in recursive_nodes(normalized):
        if not isinstance(node, dict):
            continue

        field_name = _ci_get(
            node,
            {
                "fieldName", "portName", "columnName",
                "inputFieldName", "outputFieldName"
            }
        )

        # "name" alone is too broad. Only use it when this node has
        # field-like attributes.
        has_field_attrs = any(
            _ci_get(node, {k}) is not None
            for k in (
                "dataType", "datatype", "precision", "scale",
                "direction", "portType", "fieldType"
            )
        )

        if not field_name and has_field_attrs:
            field_name = _ci_get(node, {"name"})

        if not field_name:
            continue

        transformation = _ci_get(
            node,
            {
                "transformationName", "transformName",
                "instanceName", "objectName", "parentName"
            }
        )

        # Fall back to nearby parent-ish path token.
        if not transformation:
            path_tokens = re.findall(r"\.([A-Za-z0-9_ -]+)(?:\[|\.|$)", path)
            transformation = path_tokens[-2] if len(path_tokens) >= 2 else ""

        datatype, precision, scale = _extract_type_attrs(node)

        direction = _ci_get(
            node,
            {"direction", "portType", "ioType", "fieldDirection"}
        )

        expression = _ci_get(
            node,
            {"expression", "expr", "formula", "expressionString"}
        )

        rows.append({
            "transformation": str(transformation or ""),
            "field_name": str(field_name),
            "direction": str(direction or ""),
            "datatype": datatype,
            "precision": precision,
            "scale": scale,
            "expression": _scalar_text(expression),
            "json_path": path
        })

    if not rows:
        return pd.DataFrame(
            columns=[
                "transformation", "field_name", "direction",
                "datatype", "precision", "scale",
                "expression", "json_path"
            ]
        )

    return (
        pd.DataFrame(rows)
        .drop_duplicates(
            subset=[
                "transformation", "field_name", "direction",
                "datatype", "precision", "scale", "expression"
            ]
        )
        .reset_index(drop=True)
    )


def extract_links(asset_json):
    """
    Discover explicit field-to-field mapping / connector definitions.
    """
    rows = []

    if asset_json is None:
        return pd.DataFrame()

    normalized = normalize_json(asset_json)

    for path, node in recursive_nodes(normalized):
        if not isinstance(node, dict):
            continue

        source = _ci_get(node, LINK_SOURCE_KEYS)
        target = _ci_get(node, LINK_TARGET_KEYS)

        # Explicit source+target pair is the strongest signal.
        if source is not None and target is not None:
            src = _qualified_field(source)
            tgt = _qualified_field(target)

            if src and tgt:
                rows.append({
                    "source_field": src,
                    "target_field": tgt,
                    "expression": _scalar_text(
                        _ci_get(
                            node,
                            {"expression", "expr", "formula"}
                        )
                    ),
                    "json_path": path
                })
                continue

        # Some exports use separate named keys such as
        # sourceTransformation/sourceField + targetTransformation/targetField.
        src_transform = _ci_get(
            node,
            {
                "sourceTransformation", "sourceTransform",
                "fromTransformation", "inputTransformation"
            }
        )
        src_field = _ci_get(
            node,
            {
                "sourceField", "fromField",
                "inputField", "sourcePort"
            }
        )
        tgt_transform = _ci_get(
            node,
            {
                "targetTransformation", "targetTransform",
                "toTransformation", "outputTransformation"
            }
        )
        tgt_field = _ci_get(
            node,
            {
                "targetField", "toField",
                "outputField", "targetPort"
            }
        )

        if src_field is not None and tgt_field is not None:
            src = (
                f"{src_transform}.{_scalar_text(src_field)}"
                if src_transform
                else _scalar_text(src_field)
            )
            tgt = (
                f"{tgt_transform}.{_scalar_text(tgt_field)}"
                if tgt_transform
                else _scalar_text(tgt_field)
            )

            if src and tgt:
                rows.append({
                    "source_field": src,
                    "target_field": tgt,
                    "expression": _scalar_text(
                        _ci_get(
                            node,
                            {"expression", "expr", "formula"}
                        )
                    ),
                    "json_path": path
                })

    if not rows:
        return pd.DataFrame(
            columns=[
                "source_field", "target_field",
                "expression", "json_path"
            ]
        )

    return (
        pd.DataFrame(rows)
        .drop_duplicates(
            subset=[
                "source_field", "target_field", "expression"
            ]
        )
        .reset_index(drop=True)
    )


def compare_field_definitions(fields_a, fields_b):
    key_cols = [
        "transformation",
        "field_name",
        "direction"
    ]

    attrs = [
        "datatype",
        "precision",
        "scale",
        "expression"
    ]

    def prepare(df, suffix):
        if df.empty:
            return pd.DataFrame(
                columns=key_cols + [
                    f"{a}_{suffix}"
                    for a in attrs
                ]
            )

        temp = (
            df.groupby(
                key_cols,
                dropna=False
            )[attrs]
            .agg(
                lambda s: " || ".join(
                    sorted(
                        set(
                            str(x)
                            for x in s
                            if str(x) != "nan"
                        )
                    )
                )
            )
            .reset_index()
        )

        return temp.rename(
            columns={
                a: f"{a}_{suffix}"
                for a in attrs
            }
        )

    a = prepare(fields_a, "a")
    b = prepare(fields_b, "b")

    merged = a.merge(
        b,
        on=key_cols,
        how="outer",
        indicator=True
    )

    diff_rows = []

    for _, row in merged.iterrows():
        base = {
            "transformation": row["transformation"],
            "field_name": row["field_name"],
            "direction": row["direction"]
        }

        if row["_merge"] == "left_only":
            diff_rows.append({
                **base,
                "difference_type": "FIELD_MISSING_IN_B",
                "property": "field",
                "project_a_value": "Present",
                "project_b_value": "Missing"
            })
            continue

        if row["_merge"] == "right_only":
            diff_rows.append({
                **base,
                "difference_type": "FIELD_MISSING_IN_A",
                "property": "field",
                "project_a_value": "Missing",
                "project_b_value": "Present"
            })
            continue

        for attr in attrs:
            va = str(row.get(f"{attr}_a", "") or "")
            vb = str(row.get(f"{attr}_b", "") or "")

            if va != vb:
                diff_type = {
                    "datatype": "DATATYPE_CHANGED",
                    "precision": "PRECISION_CHANGED",
                    "scale": "SCALE_CHANGED",
                    "expression": "EXPRESSION_CHANGED"
                }[attr]

                diff_rows.append({
                    **base,
                    "difference_type": diff_type,
                    "property": attr,
                    "project_a_value": va,
                    "project_b_value": vb
                })

    return pd.DataFrame(diff_rows)


def compare_field_links(links_a, links_b):
    """
    Compare source -> target mappings.

    Target field is used as the logical anchor where possible because
    the most important migration question is often:
       "Is this target column fed from the same source?"
    """
    cols = [
        "source_field",
        "target_field",
        "expression"
    ]

    if links_a.empty and links_b.empty:
        return pd.DataFrame(
            columns=[
                "target_field",
                "source_field_a",
                "source_field_b",
                "expression_a",
                "expression_b",
                "difference_type"
            ]
        )

    def collapse(df, suffix):
        if df.empty:
            return pd.DataFrame(
                columns=[
                    "target_field",
                    f"source_field_{suffix}",
                    f"expression_{suffix}"
                ]
            )

        grouped = (
            df.groupby(
                "target_field",
                dropna=False
            )
            .agg({
                "source_field": lambda s: " || ".join(
                    sorted(set(map(str, s)))
                ),
                "expression": lambda s: " || ".join(
                    sorted(
                        set(
                            str(x)
                            for x in s
                            if str(x)
                        )
                    )
                )
            })
            .reset_index()
        )

        return grouped.rename(
            columns={
                "source_field": f"source_field_{suffix}",
                "expression": f"expression_{suffix}"
            }
        )

    a = collapse(links_a, "a")
    b = collapse(links_b, "b")

    merged = a.merge(
        b,
        on="target_field",
        how="outer",
        indicator=True
    )

    def classify(row):
        if row["_merge"] == "left_only":
            return "MAPPING_MISSING_IN_B"

        if row["_merge"] == "right_only":
            return "MAPPING_MISSING_IN_A"

        src_a = str(
            row.get("source_field_a", "") or ""
        )
        src_b = str(
            row.get("source_field_b", "") or ""
        )

        exp_a = str(
            row.get("expression_a", "") or ""
        )
        exp_b = str(
            row.get("expression_b", "") or ""
        )

        if src_a != src_b:
            return "SOURCE_MAPPING_CHANGED"

        if exp_a != exp_b:
            return "MAPPING_EXPRESSION_CHANGED"

        return "SAME"

    merged["difference_type"] = merged.apply(
        classify,
        axis=1
    )

    return (
        merged[
            merged["difference_type"] != "SAME"
        ]
        .drop(columns="_merge")
        .reset_index(drop=True)
    )


# -------------------------------------------------------------------
# Run dedicated field-level comparison for every matched asset
# -------------------------------------------------------------------

field_definition_diff_frames = []
field_link_diff_frames = []
field_inventory_frames = []

for _, asset in asset_json_map.iterrows():
    json_a = asset["json_a"]
    json_b = asset["json_b"]

    if json_a is None or json_b is None:
        continue

    fields_a = extract_fields(json_a)
    fields_b = extract_fields(json_b)

    links_a = extract_links(json_a)
    links_b = extract_links(json_b)

    field_def_diff = compare_field_definitions(
        fields_a,
        fields_b
    )

    field_link_diff = compare_field_links(
        links_a,
        links_b
    )

    if not field_def_diff.empty:
        field_def_diff.insert(
            0,
            "relative_path",
            asset["relative_path"]
        )
        field_def_diff.insert(
            1,
            "asset_type",
            asset["asset_type"]
        )
        field_definition_diff_frames.append(
            field_def_diff
        )

    if not field_link_diff.empty:
        field_link_diff.insert(
            0,
            "relative_path",
            asset["relative_path"]
        )
        field_link_diff.insert(
            1,
            "asset_type",
            asset["asset_type"]
        )
        field_link_diff_frames.append(
            field_link_diff
        )

    # Useful diagnostic inventory
    for side, fields_df, links_df in (
        ("A", fields_a, links_a),
        ("B", fields_b, links_b)
    ):
        field_inventory_frames.append(
            pd.DataFrame([{
                "relative_path": asset["relative_path"],
                "asset_type": asset["asset_type"],
                "side": side,
                "field_count": len(fields_df),
                "link_count": len(links_df)
            }])
        )


field_definition_differences = (
    pd.concat(
        field_definition_diff_frames,
        ignore_index=True
    )
    if field_definition_diff_frames
    else pd.DataFrame()
)

field_link_differences = (
    pd.concat(
        field_link_diff_frames,
        ignore_index=True
    )
    if field_link_diff_frames
    else pd.DataFrame()
)

field_lineage_inventory = (
    pd.concat(
        field_inventory_frames,
        ignore_index=True
    )
    if field_inventory_frames
    else pd.DataFrame()
)

print("Field definition differences:", len(field_definition_differences))
print("Field mapping/link differences:", len(field_link_differences))

print("\nFIELD DEFINITION DIFFERENCES")
display(field_definition_differences.head(500))

print("\nFIELD MAPPING / LINEAGE DIFFERENCES")
display(field_link_differences.head(500))


## Mandatory field-lineage parser validation

**Important:** zero discovered links does **not** mean that two mappings are identical.

The notebook now labels each mapping with parser coverage:

- `PARSED_BOTH` — links were discovered in both A and B.
- `NO_LINKS_A` / `NO_LINKS_B` — one side was not parsed.
- `NO_LINKS_BOTH` — neither mapping's connector structure was parsed.
- `PAYLOAD_NOT_FOUND` — the mapping payload could not be resolved.

Only `PARSED_BOTH` is considered a reliable field-link comparison.


In [ ]:

def build_field_mapping_validation(
    asset_json_map,
    field_lineage_inventory
):
    rows = []

    counts = {}

    if not field_lineage_inventory.empty:
        for _, r in field_lineage_inventory.iterrows():
            counts[
                (
                    str(r["relative_path"]),
                    str(r["side"])
                )
            ] = {
                "field_count": int(
                    r.get("field_count", 0) or 0
                ),
                "link_count": int(
                    r.get("link_count", 0) or 0
                )
            }

    for _, asset in asset_json_map.iterrows():
        rel = str(asset["relative_path"])

        a = counts.get(
            (rel, "A"),
            {
                "field_count": 0,
                "link_count": 0
            }
        )

        b = counts.get(
            (rel, "B"),
            {
                "field_count": 0,
                "link_count": 0
            }
        )

        payload_a = asset.get(
            "json_a"
        ) is not None

        payload_b = asset.get(
            "json_b"
        ) is not None

        if not payload_a or not payload_b:
            status = "PAYLOAD_NOT_FOUND"

        elif (
            a["link_count"] == 0
            and b["link_count"] == 0
        ):
            status = "NO_LINKS_BOTH"

        elif a["link_count"] == 0:
            status = "NO_LINKS_A"

        elif b["link_count"] == 0:
            status = "NO_LINKS_B"

        else:
            status = "PARSED_BOTH"

        rows.append({
            "relative_path": rel,
            "asset_type": asset["asset_type"],
            "payload_kind_a": asset.get(
                "payload_kind_a"
            ),
            "payload_kind_b": asset.get(
                "payload_kind_b"
            ),
            "field_count_a": a["field_count"],
            "field_count_b": b["field_count"],
            "link_count_a": a["link_count"],
            "link_count_b": b["link_count"],
            "validation_status": status
        })

    return pd.DataFrame(rows)


field_mapping_validation = build_field_mapping_validation(
    asset_json_map,
    field_lineage_inventory
)

print("FIELD MAPPING PARSER VALIDATION")
display(field_mapping_validation)

unverified = field_mapping_validation[
    field_mapping_validation[
        "validation_status"
    ] != "PARSED_BOTH"
]

if not unverified.empty:
    print(
        "\nWARNING:",
        len(unverified),
        "asset(s) do not have reliable parsed field links."
    )
    print(
        "Do NOT interpret an empty field_lineage_diff "
        "as 'no difference' for these assets."
    )
    display(unverified)


## Mapping payload fallback comparison

Even if the field-link parser does not understand a particular IICS connector schema, the notebook performs a normalized structural comparison of the **actual DTEMPLATE mapping payload**.

This does not yet label every difference as a source→target link, but it prevents changed mapping definitions from being reported as identical merely because connector parsing returned zero rows.


In [ ]:

mapping_payload_summary_rows = []

for _, asset in asset_json_map.iterrows():
    ja = asset["json_a"]
    jb = asset["json_b"]

    if ja is None or jb is None:
        mapping_payload_summary_rows.append({
            "relative_path": asset["relative_path"],
            "asset_type": asset["asset_type"],
            "payload_kind_a": asset.get(
                "payload_kind_a"
            ),
            "payload_kind_b": asset.get(
                "payload_kind_b"
            ),
            "payload_structural_diff_count": None,
            "payload_status": "PAYLOAD_NOT_FOUND"
        })
        continue

    raw_diff = structural_diff(
        ja,
        jb
    )

    mapping_payload_summary_rows.append({
        "relative_path": asset["relative_path"],
        "asset_type": asset["asset_type"],
        "payload_kind_a": asset.get(
            "payload_kind_a"
        ),
        "payload_kind_b": asset.get(
            "payload_kind_b"
        ),
        "payload_structural_diff_count": len(
            raw_diff
        ),
        "payload_status": (
            "PAYLOAD_IDENTICAL"
            if raw_diff.empty
            else "PAYLOAD_DIFFERENT"
        )
    })


mapping_payload_validation = pd.DataFrame(
    mapping_payload_summary_rows
)

print("ACTUAL MAPPING PAYLOAD COMPARISON")
display(mapping_payload_validation)


## 12. Quick summary and missing transformation / mapping views

## Readable Field Mapping Difference Report

The earlier generic `field_mapping_diff` can contain large serialized JSON values.  
This section creates a much clearer report with one row per affected field/mapping.

The report focuses on:

- mapping / asset name
- transformation
- target field
- source field in Project A
- source field in Project B
- expression in Project A
- expression in Project B
- datatype / precision / scale changes
- concise difference type
- human-readable explanation


In [ ]:
# Defensive initialization for clean top-to-bottom execution
field_mapping_diff_readable = pd.DataFrame()


In [ ]:

def _safe_text(v):
    if v is None:
        return ""
    try:
        if pd.isna(v):
            return ""
    except Exception:
        pass
    return str(v)


def _split_qualified_field(value):
    value = _safe_text(value).strip()
    if not value:
        return "", ""
    if "." in value:
        left, right = value.rsplit(".", 1)
        return left, right
    return "", value


def _short_value(value, max_len=500):
    """
    Keep readable values compact. If the raw comparator contains
    serialized JSON, try to extract a useful scalar before falling
    back to a truncated representation.
    """
    text = _safe_text(value).strip()
    if not text:
        return ""

    # Try JSON decoding first.
    if text.startswith(("{", "[")):
        try:
            obj = json.loads(text)

            if isinstance(obj, dict):
                for key in (
                    "expression",
                    "fieldName",
                    "portName",
                    "columnName",
                    "name",
                    "sourceField",
                    "targetField",
                    "dataType",
                    "precision",
                    "scale",
                    "condition"
                ):
                    if key in obj and not isinstance(
                        obj[key],
                        (dict, list)
                    ):
                        return _safe_text(obj[key])

            if isinstance(obj, list) and len(obj) == 1:
                return _short_value(
                    obj[0],
                    max_len=max_len
                )

        except Exception:
            pass

    if len(text) > max_len:
        return text[:max_len] + "..."

    return text


def _infer_difference_category(
    category,
    prop,
    diff_type
):
    text = " ".join(
        [
            _safe_text(category),
            _safe_text(prop),
            _safe_text(diff_type)
        ]
    ).casefold()

    if "expression" in text:
        return "EXPRESSION"
    if "datatype" in text or "data type" in text:
        return "DATATYPE"
    if "precision" in text or "length" in text:
        return "PRECISION"
    if "scale" in text:
        return "SCALE"
    if "condition" in text:
        return "CONDITION"
    if "field" in text or "port" in text or "mapping" in text:
        return "FIELD_MAPPING"

    return "OTHER"


def _raw_semantic_to_readable(
    raw_df
):
    """
    Fallback parser for the existing generic semantic
    field_mapping_differences dataframe.

    This guarantees that field_mapping_diff_readable is not empty
    merely because the dedicated connector parser found no links.
    """
    if raw_df is None or raw_df.empty:
        return pd.DataFrame()

    rows = []

    for _, r in raw_df.iterrows():
        rel = _safe_text(
            r.get("relative_path")
        )
        asset_type = _safe_text(
            r.get("asset_type")
        )

        category = _safe_text(
            r.get("category")
        )
        name = _safe_text(
            r.get("name")
        )
        obj_type = _safe_text(
            r.get("type")
        )
        prop = _safe_text(
            r.get("property")
        )

        # Raw semantic comparator versions may use either
        # value_a/value_b or project_a_value/project_b_value.
        va = _short_value(
            r.get(
                "value_a",
                r.get("project_a_value", "")
            )
        )
        vb = _short_value(
            r.get(
                "value_b",
                r.get("project_b_value", "")
            )
        )

        raw_status = _safe_text(
            r.get(
                "status",
                r.get("difference_type", "")
            )
        ).upper()

        transformation = ""
        field_name = ""

        # Most semantic names are transformation/field-ish.
        if name:
            if "." in name:
                transformation, field_name = (
                    _split_qualified_field(name)
                )
            else:
                field_name = name

        # Try property path if name is generic.
        if not transformation and prop and "." in prop:
            p_trans, p_field = (
                _split_qualified_field(prop)
            )
            if p_trans:
                transformation = p_trans
            if not field_name:
                field_name = p_field

        if raw_status in {
            "ONLY_IN_A",
            "MISSING_IN_B"
        }:
            diff_type = "MAPPING_OR_FIELD_MISSING_IN_B"
            explanation = (
                f"'{name or prop}' exists in Project A "
                "but is missing in Project B."
            )

        elif raw_status in {
            "ONLY_IN_B",
            "MISSING_IN_A"
        }:
            diff_type = "MAPPING_OR_FIELD_MISSING_IN_A"
            explanation = (
                f"'{name or prop}' exists in Project B "
                "but is missing in Project A."
            )

        elif raw_status in {
            "CHANGED",
            "DIFFERENT"
        }:
            diff_type = "VALUE_CHANGED"
            explanation = (
                f"'{name or prop}' changed from "
                f"'{va}' in Project A to "
                f"'{vb}' in Project B."
            )

        else:
            diff_type = (
                raw_status
                if raw_status
                else "DIFFERENCE"
            )
            explanation = (
                f"Difference detected for "
                f"'{name or prop}'."
            )

        diff_category = _infer_difference_category(
            category,
            prop,
            diff_type
        )

        target_field = (
            f"{transformation}.{field_name}"
            if transformation and field_name
            else field_name
        )

        rows.append({
            "relative_path": rel,
            "asset_type": asset_type,
            "difference_category": diff_category,
            "difference_type": diff_type,
            "transformation": transformation,
            "target_field": target_field,
            "target_field_name": field_name,
            "source_field_a": "",
            "source_field_b": "",
            "expression_a": (
                va
                if diff_category == "EXPRESSION"
                else ""
            ),
            "expression_b": (
                vb
                if diff_category == "EXPRESSION"
                else ""
            ),
            "datatype_a": (
                va
                if diff_category == "DATATYPE"
                else ""
            ),
            "datatype_b": (
                vb
                if diff_category == "DATATYPE"
                else ""
            ),
            "precision_a": (
                va
                if diff_category == "PRECISION"
                else ""
            ),
            "precision_b": (
                vb
                if diff_category == "PRECISION"
                else ""
            ),
            "scale_a": (
                va
                if diff_category == "SCALE"
                else ""
            ),
            "scale_b": (
                vb
                if diff_category == "SCALE"
                else ""
            ),
            "project_a_value": va,
            "project_b_value": vb,
            "semantic_category": category,
            "semantic_property": prop,
            "explanation": explanation,
            "report_source": "RAW_SEMANTIC_FALLBACK"
        })

    return pd.DataFrame(rows)


def build_readable_field_mapping_diff(
    field_link_differences,
    field_definition_differences,
    raw_field_mapping_differences=None
):
    rows = []

    # --------------------------------------------------------------
    # A. Dedicated source -> target mapping changes
    # --------------------------------------------------------------
    if (
        field_link_differences is not None
        and not field_link_differences.empty
    ):
        for _, r in field_link_differences.iterrows():
            target_field = _safe_text(
                r.get("target_field")
            )

            tgt_transform, tgt_field_name = (
                _split_qualified_field(
                    target_field
                )
            )

            src_a = _safe_text(
                r.get("source_field_a")
            )
            src_b = _safe_text(
                r.get("source_field_b")
            )

            exp_a = _safe_text(
                r.get("expression_a")
            )
            exp_b = _safe_text(
                r.get("expression_b")
            )

            diff_type = _safe_text(
                r.get("difference_type")
            )

            if diff_type == "MAPPING_MISSING_IN_B":
                explanation = (
                    f"Target field '{target_field}' is mapped in "
                    f"Project A from '{src_a}', but the mapping "
                    "is missing in Project B."
                )
            elif diff_type == "MAPPING_MISSING_IN_A":
                explanation = (
                    f"Target field '{target_field}' is mapped in "
                    f"Project B from '{src_b}', but the mapping "
                    "is missing in Project A."
                )
            elif diff_type == "SOURCE_MAPPING_CHANGED":
                explanation = (
                    f"Target field '{target_field}' receives data "
                    f"from '{src_a}' in Project A but from "
                    f"'{src_b}' in Project B."
                )
            elif diff_type == "MAPPING_EXPRESSION_CHANGED":
                explanation = (
                    f"Expression feeding '{target_field}' changed "
                    "between Project A and Project B."
                )
            else:
                explanation = (
                    f"Field mapping difference detected for "
                    f"'{target_field}'."
                )

            rows.append({
                "relative_path": _safe_text(
                    r.get("relative_path")
                ),
                "asset_type": _safe_text(
                    r.get("asset_type")
                ),
                "difference_category": "FIELD_MAPPING",
                "difference_type": diff_type,
                "transformation": tgt_transform,
                "target_field": target_field,
                "target_field_name": tgt_field_name,
                "source_field_a": src_a,
                "source_field_b": src_b,
                "expression_a": exp_a,
                "expression_b": exp_b,
                "datatype_a": "",
                "datatype_b": "",
                "precision_a": "",
                "precision_b": "",
                "scale_a": "",
                "scale_b": "",
                "project_a_value": src_a or exp_a,
                "project_b_value": src_b or exp_b,
                "semantic_category": "",
                "semantic_property": "",
                "explanation": explanation,
                "report_source": "DEDICATED_LINEAGE"
            })

    # --------------------------------------------------------------
    # B. Dedicated field-definition differences
    # --------------------------------------------------------------
    if (
        field_definition_differences is not None
        and not field_definition_differences.empty
    ):
        for _, r in field_definition_differences.iterrows():
            transformation = _safe_text(
                r.get("transformation")
            )
            field_name = _safe_text(
                r.get("field_name")
            )
            diff_type = _safe_text(
                r.get("difference_type")
            )
            va = _short_value(
                r.get("project_a_value")
            )
            vb = _short_value(
                r.get("project_b_value")
            )

            category = _infer_difference_category(
                "",
                r.get("property"),
                diff_type
            )

            target_field = (
                f"{transformation}.{field_name}"
                if transformation
                else field_name
            )

            rows.append({
                "relative_path": _safe_text(
                    r.get("relative_path")
                ),
                "asset_type": _safe_text(
                    r.get("asset_type")
                ),
                "difference_category": category,
                "difference_type": diff_type,
                "transformation": transformation,
                "target_field": target_field,
                "target_field_name": field_name,
                "source_field_a": "",
                "source_field_b": "",
                "expression_a": (
                    va
                    if category == "EXPRESSION"
                    else ""
                ),
                "expression_b": (
                    vb
                    if category == "EXPRESSION"
                    else ""
                ),
                "datatype_a": (
                    va
                    if category == "DATATYPE"
                    else ""
                ),
                "datatype_b": (
                    vb
                    if category == "DATATYPE"
                    else ""
                ),
                "precision_a": (
                    va
                    if category == "PRECISION"
                    else ""
                ),
                "precision_b": (
                    vb
                    if category == "PRECISION"
                    else ""
                ),
                "scale_a": (
                    va
                    if category == "SCALE"
                    else ""
                ),
                "scale_b": (
                    vb
                    if category == "SCALE"
                    else ""
                ),
                "project_a_value": va,
                "project_b_value": vb,
                "semantic_category": "",
                "semantic_property": _safe_text(
                    r.get("property")
                ),
                "explanation": (
                    f"{diff_type}: '{target_field}' differs "
                    "between Project A and Project B."
                ),
                "report_source": "DEDICATED_FIELD_DEFINITION"
            })

    dedicated_df = pd.DataFrame(rows)

    # --------------------------------------------------------------
    # C. Raw semantic fallback
    # --------------------------------------------------------------
    fallback_df = _raw_semantic_to_readable(
        raw_field_mapping_differences
    )

    if dedicated_df.empty:
        result = fallback_df.copy()
    elif fallback_df.empty:
        result = dedicated_df.copy()
    else:
        # Keep both. Dedicated rows are more precise; raw semantic
        # rows ensure we never lose differences not understood by
        # the dedicated parser.
        result = pd.concat(
            [dedicated_df, fallback_df],
            ignore_index=True,
            sort=False
        )

    columns = [
        "relative_path",
        "asset_type",
        "difference_category",
        "difference_type",
        "transformation",
        "target_field",
        "target_field_name",
        "source_field_a",
        "source_field_b",
        "expression_a",
        "expression_b",
        "datatype_a",
        "datatype_b",
        "precision_a",
        "precision_b",
        "scale_a",
        "scale_b",
        "project_a_value",
        "project_b_value",
        "semantic_category",
        "semantic_property",
        "explanation",
        "report_source"
    ]

    if result.empty:
        return pd.DataFrame(
            columns=columns
        )

    for col in columns:
        if col not in result.columns:
            result[col] = ""

    result = (
        result[columns]
        .drop_duplicates()
        .sort_values(
            [
                "relative_path",
                "transformation",
                "target_field",
                "difference_category",
                "difference_type"
            ],
            na_position="last"
        )
        .reset_index(drop=True)
    )

    return result


field_mapping_diff_readable = (
    build_readable_field_mapping_diff(
        field_link_differences,
        field_definition_differences,
        field_mapping_differences
    )
)

print(
    "Readable field mapping differences:",
    len(field_mapping_diff_readable)
)

if field_mapping_diff_readable.empty:
    print(
        "No readable field differences were found from either "
        "the dedicated parser or the raw semantic comparator."
    )
else:
    print(
        "\nReport source counts:"
    )
    display(
        field_mapping_diff_readable[
            "report_source"
        ]
        .value_counts(
            dropna=False
        )
        .rename_axis(
            "report_source"
        )
        .reset_index(
            name="count"
        )
    )

display(
    field_mapping_diff_readable.head(1000)
)


In [ ]:
if 'field_mapping_diff_readable' not in globals():
    field_mapping_diff_readable = pd.DataFrame()

print("=" * 90)
print("IICS PROJECT COMPARISON")
print("=" * 90)

print("Project A                    :", PROJECT_A)
print("Folder A                     :", FOLDER_A or "<project root>")
print("Comparison root A            :", ROOT_A)
print("Project B                    :", PROJECT_B)
print("Folder B                     :", FOLDER_B or "<project root>")
print("Comparison root B            :", ROOT_B)
print()
print(
    f"Assets in Folder A           : {len(inventory_a):,}"
)
print(
    f"Assets in Folder B           : {len(inventory_b):,}"
)
print(
    f"Common assets                : {len(common_assets):,}"
)
print(
    f"Only in Project A            : {len(only_a):,}"
)
print(
    f"Only in Project B            : {len(only_b):,}"
)
print(
    f"Deep-compared common assets  : {len(asset_comparison_summary):,}"
)

if not asset_comparison_summary.empty:
    print(
        "Identical assets             :",
        int(
            (
                asset_comparison_summary[
                    "comparison_status"
                ]
                == "IDENTICAL"
            ).sum()
        )
    )
    print(
        "Different assets             :",
        int(
            (
                asset_comparison_summary[
                    "comparison_status"
                ]
                == "DIFFERENT"
            ).sum()
        )
    )
    print(
        "JSON not automatically found :",
        int(
            (
                asset_comparison_summary[
                    "comparison_status"
                ]
                == "JSON_NOT_FOUND"
            ).sum()
        )
    )

if not semantic_differences.empty:
    print("\nSemantic difference summary")
    display(
        semantic_differences
        .groupby(
            [
                "category",
                "change_type"
            ],
            dropna=False
        )
        .size()
        .reset_index(
            name="count"
        )
        .sort_values(
            "count",
            ascending=False
        )
    )

print("\nTRANSFORMATION DIFFERENCES")
display(
    transformation_differences.head(300)
)

print("\nFIELD / MAPPING / EXPRESSION / CONDITION DIFFERENCES")
display(
    field_mapping_differences.head(500)
)


print("\nDEDICATED FIELD-LINEAGE SUMMARY")
print(
    "Field definition differences :",
    len(field_definition_differences)
)
print(
    "Field mapping/link differences:",
    len(field_link_differences)
)

if not field_link_differences.empty:
    display(
        field_link_differences
        .groupby(
            "difference_type",
            dropna=False
        )
        .size()
        .reset_index(name="count")
        .sort_values(
            "count",
            ascending=False
        )
    )


print(
    "\nReadable field mapping differences:",
    len(field_mapping_diff_readable)
)

if not field_mapping_diff_readable.empty:
    display(
        field_mapping_diff_readable
        .groupby(
            [
                "difference_category",
                "difference_type"
            ],
            dropna=False
        )
        .size()
        .reset_index(name="count")
        .sort_values(
            "count",
            ascending=False
        )
    )


## 13. Export CSV and Excel reports

## Target Properties Comparison

This section compares target-level configuration between Project A and Project B.

It extracts and compares target connection, target object/table, operation, write disposition/load mode, update mode/update strategy, pre-SQL, post-SQL, truncate behavior, insert/update/delete/upsert flags, bulk/write options, commit/batch settings, reject/error handling, and related target properties.

The output is one concise row per changed target property.


In [ ]:

TARGET_PROPERTY_ALIASES = {
    "connection": {
        "connection", "connectionname", "connectionref", "connectionreference",
        "targetconnection", "connectionid", "connectionpath"
    },
    "target_object": {
        "targetobject", "targettablename", "tablename", "table", "objectname",
        "targetname", "targettable", "object"
    },
    "operation": {
        "operation", "targetoperation", "writeoperation", "operationtype"
    },
    "write_disposition": {
        "writedisposition", "loadtype", "loadmode", "writemode"
    },
    "update_mode": {
        "updatemode", "updatestrategy", "updateoption"
    },
    "pre_sql": {
        "presql", "presqloverride"
    },
    "post_sql": {
        "postsql", "postsqloverride"
    },
    "truncate_before_load": {
        "truncate", "truncatebeforeload", "truncatetarget"
    },
    "insert_enabled": {
        "insert", "insertenabled", "allowinsert"
    },
    "update_enabled": {
        "update", "updateenabled", "allowupdate"
    },
    "delete_enabled": {
        "delete", "deleteenabled", "allowdelete"
    },
    "upsert_enabled": {
        "upsert", "upsertenabled"
    },
    "bulk_mode": {
        "bulkmode", "bulkwrite"
    },
    "commit_interval": {
        "commitinterval", "commitcount", "batchsize"
    },
    "reject_handling": {
        "rejecthandling", "errorhandling", "rejectfilename"
    }
}

def _norm_key(k):
    return re.sub(r"[^a-z0-9]", "", str(k).casefold())

TARGET_PROPERTY_LOOKUP = {}
for canonical, aliases in TARGET_PROPERTY_ALIASES.items():
    TARGET_PROPERTY_LOOKUP[_norm_key(canonical)] = canonical
    for alias in aliases:
        TARGET_PROPERTY_LOOKUP[_norm_key(alias)] = canonical

def _looks_like_target_node(path, node):
    typ = _ci_get(
        node,
        {"type", "transformationType", "transformType", "objectType"}
    )
    if typ and "target" in str(typ).casefold():
        return True

    text = f"{path} {compact_json(node)[:1200]}".casefold()
    return any(
        token in text
        for token in (
            "targettransformation",
            "target transformation",
            "targetobject",
            "target object"
        )
    )

def _flatten_scalars(obj, prefix=""):
    rows = []
    if isinstance(obj, dict):
        for k, v in obj.items():
            p = f"{prefix}.{k}" if prefix else str(k)
            if isinstance(v, (dict, list)):
                rows.extend(_flatten_scalars(v, p))
            else:
                rows.append((p, k, v))
    elif isinstance(obj, list):
        for i, v in enumerate(obj):
            p = f"{prefix}[{i}]"
            if isinstance(v, (dict, list)):
                rows.extend(_flatten_scalars(v, p))
            else:
                rows.append((p, str(i), v))
    return rows

def _target_identity(path, node):
    for keys in (
        {"targetName", "targetObject", "targetTableName"},
        {"transformationName", "instanceName", "objectName"},
        {"name"}
    ):
        v = _ci_get(node, keys)
        if isinstance(v, (str, int, float)) and str(v).strip():
            return str(v).strip()
    return "TARGET"

def extract_target_properties(asset_json):
    columns = [
        "target_name", "property_name", "property_value", "json_path"
    ]
    if asset_json is None:
        return pd.DataFrame(columns=columns)

    normalized = normalize_json(asset_json)
    rows = []

    for path, node in recursive_nodes(normalized):
        if not isinstance(node, dict):
            continue
        if not _looks_like_target_node(path, node):
            continue

        target_name = _target_identity(path, node)

        for scalar_path, raw_key, raw_value in _flatten_scalars(node):
            canonical = TARGET_PROPERTY_LOOKUP.get(_norm_key(raw_key))
            if not canonical:
                continue

            rows.append({
                "target_name": target_name,
                "property_name": canonical,
                "property_value": _short_value(raw_value, 1000),
                "json_path": f"{path}.{scalar_path}"
            })

    if not rows:
        return pd.DataFrame(columns=columns)

    df = pd.DataFrame(rows)
    return (
        df.groupby(
            ["target_name", "property_name"],
            dropna=False
        )
        .agg({
            "property_value": lambda s: " || ".join(
                sorted(set(_safe_text(x) for x in s if _safe_text(x)))
            ),
            "json_path": lambda s: " || ".join(sorted(set(map(str, s))))
        })
        .reset_index()
    )

def compare_target_properties(props_a, props_b):
    cols = [
        "target_name", "property_name", "project_a_value",
        "project_b_value", "difference_type", "explanation"
    ]

    if props_a.empty and props_b.empty:
        return pd.DataFrame(columns=cols)

    a = props_a[["target_name", "property_name", "property_value"]].rename(
        columns={"property_value": "project_a_value"}
    )
    b = props_b[["target_name", "property_name", "property_value"]].rename(
        columns={"property_value": "project_b_value"}
    )

    merged = a.merge(
        b,
        on=["target_name", "property_name"],
        how="outer",
        indicator=True
    )

    out = []
    for _, r in merged.iterrows():
        va = _safe_text(r.get("project_a_value"))
        vb = _safe_text(r.get("project_b_value"))

        if r["_merge"] == "left_only":
            diff = "PROPERTY_MISSING_IN_B"
            explanation = (
                f"Target property '{r['property_name']}' for '{r['target_name']}' "
                "exists in Project A but is missing in Project B."
            )
        elif r["_merge"] == "right_only":
            diff = "PROPERTY_MISSING_IN_A"
            explanation = (
                f"Target property '{r['property_name']}' for '{r['target_name']}' "
                "exists in Project B but is missing in Project A."
            )
        elif va != vb:
            diff = "PROPERTY_CHANGED"
            explanation = (
                f"Target property '{r['property_name']}' for '{r['target_name']}' "
                f"changed from '{va}' to '{vb}'."
            )
        else:
            continue

        out.append({
            "target_name": r["target_name"],
            "property_name": r["property_name"],
            "project_a_value": va,
            "project_b_value": vb,
            "difference_type": diff,
            "explanation": explanation
        })

    return pd.DataFrame(out, columns=cols)

target_property_diff_frames = []
target_property_inventory_frames = []

for _, asset in asset_json_map.iterrows():
    props_a = extract_target_properties(asset.get("json_a"))
    props_b = extract_target_properties(asset.get("json_b"))

    diff = compare_target_properties(props_a, props_b)

    if not diff.empty:
        diff.insert(0, "relative_path", asset["relative_path"])
        diff.insert(1, "asset_type", asset["asset_type"])
        target_property_diff_frames.append(diff)

    target_property_inventory_frames.append(
        pd.DataFrame([
            {
                "relative_path": asset["relative_path"],
                "asset_type": asset["asset_type"],
                "side": "A",
                "target_property_count": len(props_a),
                "target_count": props_a["target_name"].nunique() if not props_a.empty else 0
            },
            {
                "relative_path": asset["relative_path"],
                "asset_type": asset["asset_type"],
                "side": "B",
                "target_property_count": len(props_b),
                "target_count": props_b["target_name"].nunique() if not props_b.empty else 0
            }
        ])
    )

target_properties_diff = (
    pd.concat(target_property_diff_frames, ignore_index=True)
    if target_property_diff_frames
    else pd.DataFrame(
        columns=[
            "relative_path", "asset_type", "target_name", "property_name",
            "project_a_value", "project_b_value", "difference_type", "explanation"
        ]
    )
)

target_properties_inventory = (
    pd.concat(target_property_inventory_frames, ignore_index=True)
    if target_property_inventory_frames
    else pd.DataFrame()
)

if not target_properties_diff.empty:
    target_properties_summary = (
        target_properties_diff
        .groupby(
            ["relative_path", "property_name", "difference_type"],
            dropna=False
        )
        .size()
        .reset_index(name="difference_count")
        .sort_values(
            ["relative_path", "difference_count"],
            ascending=[True, False]
        )
    )
else:
    target_properties_summary = pd.DataFrame(
        columns=[
            "relative_path", "property_name", "difference_type", "difference_count"
        ]
    )

print("Target property differences:", len(target_properties_diff))
display(target_properties_diff.head(1000))


In [ ]:
if 'field_mapping_diff_readable' not in globals():
    field_mapping_diff_readable = pd.DataFrame()

report_dir = (
    Path(OUTPUT_DIR)
    / "reports"
)
report_dir.mkdir(
    parents=True,
    exist_ok=True
)

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

reports = {
    "target_properties_diff": target_properties_diff,
    "target_properties_summary": target_properties_summary,
    "target_properties_inventory": target_properties_inventory,
    "inventory_folder_a": inventory_a,
    "inventory_folder_b": inventory_b,
    "common_assets": common_assets,
    "only_folder_a": only_a,
    "only_folder_b": only_b,
    "comparison_summary": asset_comparison_summary,
    "transformation_diff": transformation_differences,
    "field_mapping_diff_raw": field_mapping_differences,
    "field_mapping_diff": field_mapping_diff_readable,
    "field_definition_diff": field_definition_differences,
    "field_lineage_diff": field_link_differences,
    "field_lineage_inventory": field_lineage_inventory,
    "field_mapping_validation": field_mapping_validation,
    "mapping_payload_validation": mapping_payload_validation,
    "semantic_diff": semantic_differences,
    "structural_diff": structural_differences,
    "json_resolution": asset_json_map.drop(
        columns=[
            "json_a",
            "json_b"
        ],
        errors="ignore"
    )
}

for name, df in reports.items():
    df.to_csv(
        report_dir /
        f"{name}_{timestamp}.csv",
        index=False
    )

excel_path = (
    report_dir /
    f"iics_folder_comparison_{timestamp}.xlsx"
)

with pd.ExcelWriter(
    excel_path,
    engine="openpyxl"
) as writer:
    for name, df in reports.items():
        df.to_excel(
            writer,
            sheet_name=name[:31],
            index=False
        )

print("Reports folder:", report_dir)
print("Excel report  :", excel_path)


## 14. Optional: inspect one asset

Set `ASSET_TO_INSPECT` to a relative path shown in the summary, for example:

```python
ASSET_TO_INSPECT = "Mappings/m_Claims_Load"
```


In [ ]:
ASSET_TO_INSPECT = None

if ASSET_TO_INSPECT:
    if not semantic_differences.empty:
        print("Semantic differences")
        display(
            semantic_differences[
                semantic_differences[
                    "relative_path"
                ].str.casefold()
                ==
                ASSET_TO_INSPECT.casefold()
            ]
        )

    if not structural_differences.empty:
        print("Structural differences")
        display(
            structural_differences[
                structural_differences[
                    "relative_path"
                ].str.casefold()
                ==
                ASSET_TO_INSPECT.casefold()
            ]
        )


## Interpretation

The most useful reports are:

- **comparison_summary** — one row per common asset.
- **transformation_diff** — missing, extra, or changed transformations.
- **field_mapping_diff** — field/port mappings, expressions, filters, joins, lookup conditions, and similar mapping logic.
- **only_project_a / only_project_b** — assets missing from one project.
- **structural_diff** — exhaustive normalized JSON differences when a semantic property is not recognized.

`ONLY_IN_A` means the component exists in Project A but not Project B.  
`ONLY_IN_B` means the component exists in Project B but not Project A.  
`CHANGED` means the logical component is present in both but its definition differs.


## Folder matching example

With:

```python
PROJECT_A = "DEV"
FOLDER_A = "Claims/Inbound"

PROJECT_B = "PROD"
FOLDER_B = "Claims/Inbound"
```

these assets are matched:

```text
DEV/Claims/Inbound/Mappings/m_claim_load
PROD/Claims/Inbound/Mappings/m_claim_load
```

because their path relative to the selected folder root is identical:

```text
Mappings/m_claim_load
```

Nested subfolders are included automatically.


## Dedicated field-level result interpretation

The enhanced notebook now produces two important additional reports:

### `field_definition_diff`

Detects:

- `FIELD_MISSING_IN_A`
- `FIELD_MISSING_IN_B`
- `DATATYPE_CHANGED`
- `PRECISION_CHANGED`
- `SCALE_CHANGED`
- `EXPRESSION_CHANGED`

### `field_lineage_diff`

Detects:

- `MAPPING_MISSING_IN_A`
- `MAPPING_MISSING_IN_B`
- `SOURCE_MAPPING_CHANGED`
- `MAPPING_EXPRESSION_CHANGED`

Example:

| Target field | Project A source | Project B source | Difference |
|---|---|---|---|
| TGT_CLAIM.CLAIM_ID | SRC.CLAIM_ID | SRC.CLAIM_ID | Same |
| TGT_CLAIM.POLICY_NO | EXP.POLICY_NO | Missing | MAPPING_MISSING_IN_B |
| TGT_CLAIM.CLAIM_AMT | SRC.CLAIM_AMT | EXP.CLAIM_AMT | SOURCE_MAPPING_CHANGED |

The `field_lineage_inventory` report shows how many fields and links were discovered from each exported mapping. If an asset shows `0` links, inspect its export JSON because that IICS asset/release may use a connector structure not yet covered by the parser.


### Readable report diagnostics

This diagnostic shows whether differences came from the dedicated lineage parser or from the raw semantic fallback.  
If `RAW_SEMANTIC_FALLBACK` dominates, the notebook is still detecting differences, but your IICS export structure is not yet fully decoded into source→target lineage.


In [ ]:

print("Dedicated field-link rows :", len(field_link_differences))
print("Dedicated field-def rows  :", len(field_definition_differences))
print("Raw field semantic rows   :", len(field_mapping_differences))
print("Readable output rows       :", len(field_mapping_diff_readable))

if not field_mapping_diff_readable.empty:
    display(
        field_mapping_diff_readable[
            [
                "relative_path",
                "difference_category",
                "difference_type",
                "transformation",
                "target_field",
                "project_a_value",
                "project_b_value",
                "report_source",
                "explanation"
            ]
        ].head(500)
    )


## How to judge whether field mapping results are trustworthy

Check `field_mapping_validation` **before** checking `field_lineage_diff`.

A mapping is safely comparable at the field-link level only when:

```text
validation_status = PARSED_BOTH
```

For DTEMPLATE mappings, also check:

```text
payload_kind_a = MAPPING_BIN
payload_kind_b = MAPPING_BIN
```

If you see:

```text
NO_LINKS_BOTH
```

the notebook is no longer allowed to call that mapping "same." It means the IICS payload was found, but the current generic connector parser did not recognize that POD's connector structure.

The `mapping_payload_validation` report will still tell you whether the underlying DTEMPLATE payload is structurally different.


## Which Excel sheet should you use?

Use **`field_mapping_diff`** as the primary business-readable report.

It now gives concise rows such as:

| Mapping | Target Field | Source A | Source B | Difference |
|---|---|---|---|---|
| m_claim_load | TGT_CLAIM.POLICY_NO | EXP.POLICY_NO |  | MAPPING_MISSING_IN_B |
| m_claim_load | TGT_CLAIM.CLAIM_AMT | SRC.CLAIM_AMT | EXP.CLAIM_AMT | SOURCE_MAPPING_CHANGED |
| m_claim_load | EXP.NET_AMT |  |  | EXPRESSION_CHANGED |

The old exhaustive JSON-oriented comparison is preserved as **`field_mapping_diff_raw`** only for technical diagnostics.


## Target properties output

Use `target_properties_diff` to review differences in connection, operation, write disposition/load mode, update mode, pre-SQL, post-SQL, truncate behavior, insert/update/delete/upsert settings, and related target write properties.

Example output:

| Mapping | Target | Property | Project A | Project B | Difference |
|---|---|---|---|---|---|
| m_claim_load | TGT_CLAIM | connection | DEV_AURORA | PROD_AURORA | PROPERTY_CHANGED |
| m_claim_load | TGT_CLAIM | operation | Insert | Upsert | PROPERTY_CHANGED |
| m_claim_load | TGT_CLAIM | pre_sql | TRUNCATE TABLE ... |  | PROPERTY_MISSING_IN_B |
